In [ ]:
Converting_model_ternarywithLoRA in Colab

https://claude.ai/chat/1a6e1478-43d4-44fd-96f2-3e00f786b660

https://huggingface.co/models?other=base_model:adapter:Qwen/Qwen2.5-0.5B-Instruct

In [1]:
!pip -q install -U transformers peft accelerate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # صغير وسريع على CPU
GROUP = 64      # حجم المجموعة لكل scale (جرّب 32 لجودة أعلى)
RANK  = 64      # رتبة LoRA التعويضية
torch.set_num_threads(torch.get_num_threads())

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32).eval()

def gen(m, prompt, n=60):
    text = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                   tokenize=False, add_generation_prompt=True)
    x = tok(text, return_tensors="pt")
    with torch.no_grad():
        y = m.generate(**x, max_new_tokens=n, do_sample=False, repetition_penalty=1.1)
    return tok.decode(y[0][x.input_ids.shape[1]:], skip_special_tokens=True)

PROMPT = "اشرح باختصار ما هو الذكاء الاصطناعي؟"
print("== الأصلي ==\n", gen(model, PROMPT))

# ---------- 1) التحويل إلى ثلاثي ----------
def ternarize(W, group=GROUP, iters=8):
    out, inp = W.shape
    assert inp % group == 0, "غيّر GROUP ليقسم أبعاد النموذج"
    Wg = W.reshape(out, -1, group)
    a = Wg.abs().mean(-1, keepdim=True)
    for _ in range(iters):                      # تحسين scale بالتناوب
        t = (Wg / a).round().clamp(-1, 1)
        a = (Wg * t).sum(-1, keepdim=True) / (t * t).sum(-1, keepdim=True).clamp(min=1)
    t = (Wg / a).round().clamp(-1, 1)
    return (t * a).reshape(out, inp)

factors = {}
for name, m in model.named_modules():
    if isinstance(m, torch.nn.Linear) and ".layers." in name:   # لا نلمس embed ولا lm_head
        W = m.weight.data
        Wq = ternarize(W)
        U, S, V = torch.svd_lowrank(W - Wq, q=RANK, niter=4)    # خطأ التكميم ≈ U S Vᵀ
        s = S.sqrt()
        factors[name] = ((U * s).contiguous(), (V * s).T.contiguous())  # (A: out×r, B: r×in)
        m.weight.data = Wq

print("== ثلاثي فقط ==\n", gen(model, PROMPT))

# ---------- 2) LoRA جاهزة (من خطأ التكميم) بدون دمج ----------
cfg = LoraConfig(r=RANK, lora_alpha=RANK, lora_dropout=0.0, bias="none",
                 target_modules=["q_proj","k_proj","v_proj","o_proj",
                                 "gate_proj","up_proj","down_proj"])
pm = get_peft_model(model, cfg).eval()

for name, m in pm.named_modules():
    if hasattr(m, "lora_A") and "default" in m.lora_A:
        A, B = factors[name.replace("base_model.model.", "")]
        m.lora_A["default"].weight.data = B.clone()   # r × in
        m.lora_B["default"].weight.data = A.clone()   # out × r

print("== ثلاثي + LoRA (غير مدموجة) ==\n", gen(pm, PROMPT))

with pm.disable_adapter():                             # للمقارنة
    print("== نفس النموذج بتعطيل LoRA ==\n", gen(pm, PROMPT))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 18.2 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

== الأصلي ==
 الذكاء الاصطناعي (AI) هو فرع من العلوم الحاسوبية يركز على إنشاء وتطوير أجهزة ومفاهيم يمكنها تعلم وتتطور بنفسها دون الحاجة إلى التوجيه أو التعليم. إليك بعض النقاط
== ثلاثي فقط ==
 




ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [ ]:
pm.load_adapter("اسم/المستودع", adapter_name="ext")
pm.set_adapter("ext")                  # تشغيلها وحدها

# أو تشغيلها مع LoRA التعويض معًا، بدون دمج في الأوزان:
pm.add_weighted_adapter(["default", "ext"], [1.0, 1.0],
                        adapter_name="both", combination_type="cat")
pm.set_adapter("both")
print(gen(pm, PROMPT))

In [ ]:
ملاحظات صريحة
لا يوجد سحر بدون تدريب. التحويل الثلاثي لنموذج مدرَّب بدقة كاملة يدمّر جزءًا كبيرًا من جودته. LoRA الـ SVD تسترجع جزءًا من الخطأ فقط، وتتحسن الجودة بزيادة RANK أو تصغير GROUP، لكن لن يصل الناتج لجودة النموذج الأصلي.
LoRA مدرَّبة على النموذج الكامل تعطي فائدة محدودة على الأوزان الثلاثية، لأنها لم تُدرَّب على خطأ التكميم. LoRA تعويضية من الخطأ نفسه (كما في الكود) أنفع هنا.
لجودة أفضل: أبقِ طبقات حساسة بدقة أعلى (مثل v_proj وdown_proj أو أول وآخر طبقتين) بإخراجها من حلقة ternarize.
النموذج مخزَّن هنا كقيم ثلاثية داخل fp32 للتجربة فقط، فلا يوفّر ذاكرة. التوفير الفعلي يحتاج تعبئة الأوزان (2 bit) ونواة تشغيل خاصة.

لو أردت، أعدّل الكود ليستثني طبقات معيّنة، أو يقيس الـ perplexity بدل النظر في النصوص فقط.

In [ ]:
EXT = "taronklm/Qwen2.5-0.5B-Instruct-lora-chatbot"
pm.load_adapter(EXT, adapter_name="ext")

# 1) الـ LoRA الجاهزة وحدها فوق الثلاثي (غير مدموجة)
pm.set_adapter("ext")
print("== ثلاثي + LoRA الجاهزة ==\n", gen(pm, PROMPT))

# 2) LoRA التعويض وحدها (للمقارنة)
pm.set_adapter("default")
print("== ثلاثي + LoRA التعويض ==\n", gen(pm, PROMPT))

# 3) الاثنتان معًا بدون دمج في الأوزان
pm.add_weighted_adapter(["default", "ext"], [1.0, 1.0],
                        adapter_name="both", combination_type="cat")
pm.set_adapter("both")
print("== ثلاثي + الاثنتان ==\n", gen(pm, PROMPT))

In [ ]:
اخترت لك taronklm/Qwen2.5-0.5B-Instruct-lora-chatbot. هي الوحيدة في أول القائمة التي تبدو عامة الغرض (شات بوت) ومبنية على نفس النموذج Qwen2.5-0.5B-Instruct الذي نستخدمه في الكود. باقي القائمة متخصص في مهام لا تفيدك:

Jailbreak-Detector: كاشف محاولات الاختراق.
careerbot: مساعد وظائف.
qwen-sms وqwen-email: لكتابة رسائل SMS وإيميلات.
Reward: نماذج مكافأة، وليست للمحادثة.

بطاقتها على Hugging Face لا تذكر بيانات التدريب ("unknown dataset")، لذلك لا أضمن أنها ستحسّن النموذج الثلاثي، وهي لم تُدرَّب على تعويض خطأ التكميم. جرّبها بالطريقة التالية بعد كود الرد السابق، أي بعد تعريف pm وgen وPROMPT

In [2]:
!pip -q uninstall -y torchao

In [3]:
!pip -q uninstall -y torchao
!pip -q install -U transformers peft accelerate

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

# ================== الإعدادات ==================
MODEL   = "Qwen/Qwen2.5-0.5B-Instruct"
EXT_LORA = "taronklm/Qwen2.5-0.5B-Instruct-lora-chatbot"
GROUP   = 32
RANK    = 128
TARGETS = ("gate_proj", "up_proj", "down_proj")   # أضف "q_proj","k_proj","o_proj" تدريجيًا
PROMPT  = "اشرح باختصار ما هو الذكاء الاصطناعي؟"
EVAL_TEXT = ("الذكاء الاصطناعي هو مجال في علوم الحاسوب يهتم ببناء أنظمة قادرة على التعلم "
             "واتخاذ القرارات. Artificial intelligence is a field of computer science "
             "that builds systems able to learn from data and make decisions.")
# ===============================================

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).eval()

def gen(m, prompt=PROMPT, n=60):
    text = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                   tokenize=False, add_generation_prompt=True)
    x = tok(text, return_tensors="pt")
    with torch.no_grad():
        y = m.generate(**x, max_new_tokens=n, do_sample=False, repetition_penalty=1.1)
    return repr(tok.decode(y[0][x.input_ids.shape[1]:], skip_special_tokens=True))

def ppl(m, text=EVAL_TEXT):
    x = tok(text, return_tensors="pt")
    with torch.no_grad():
        loss = m(**x, labels=x.input_ids).loss
    return torch.exp(loss).item()

print("== الأصلي ==")
print("ppl:", round(ppl(model), 2)); print(gen(model))

# ---------- 1) التحويل إلى ثلاثي ----------
def ternarize(W, group=GROUP, iters=8):
    out, inp = W.shape
    assert inp % group == 0, "غيّر GROUP ليقسم أبعاد النموذج"
    Wg = W.reshape(out, -1, group)
    a = Wg.abs().mean(-1, keepdim=True)
    for _ in range(iters):
        t = (Wg / a).round().clamp(-1, 1)
        a = (Wg * t).sum(-1, keepdim=True) / (t * t).sum(-1, keepdim=True).clamp(min=1)
    t = (Wg / a).round().clamp(-1, 1)
    return (t * a).reshape(out, inp)

factors = {}
for name, m in model.named_modules():
    if isinstance(m, torch.nn.Linear) and ".layers." in name and name.endswith(TARGETS):
        W = m.weight.data
        Wq = ternarize(W)
        U, S, V = torch.svd_lowrank(W - Wq, q=RANK, niter=4)   # خطأ التكميم ≈ U S Vᵀ
        s = S.sqrt()
        factors[name] = ((U * s).contiguous(), (V * s).T.contiguous())  # A: out×r , B: r×in
        m.weight.data = Wq
print(f"تم تحويل {len(factors)} طبقة إلى ثلاثي")

print("\n== ثلاثي فقط ==")
print("ppl:", round(ppl(model), 2)); print(gen(model))

# ---------- 2) LoRA التعويض (جاهزة بدون تدريب، غير مدموجة) ----------
cfg = LoraConfig(r=RANK, lora_alpha=RANK, lora_dropout=0.0, bias="none",
                 target_modules=list(TARGETS))
pm = get_peft_model(model, cfg).eval()

for name, m in pm.named_modules():
    if hasattr(m, "lora_A") and "default" in m.lora_A:
        A, B = factors[name.replace("base_model.model.", "")]
        m.lora_A["default"].weight.data = B.clone()   # r × in
        m.lora_B["default"].weight.data = A.clone()   # out × r

pm.set_adapter("default")
print("\n== ثلاثي + LoRA التعويض ==")
print("ppl:", round(ppl(pm), 2)); print(gen(pm))

# ---------- 3) LoRA الجاهزة من Hugging Face (غير مدموجة) ----------
try:
    pm.load_adapter(EXT_LORA, adapter_name="ext")
    pm.set_adapter("ext")
    print("\n== ثلاثي + LoRA الجاهزة ==")
    print("ppl:", round(ppl(pm), 2)); print(gen(pm))

    pm.add_weighted_adapter(["default", "ext"], [1.0, 1.0],
                            adapter_name="both", combination_type="cat")
    pm.set_adapter("both")
    print("\n== ثلاثي + الاثنتان ==")
    print("ppl:", round(ppl(pm), 2)); print(gen(pm))
except Exception as e:
    print("\nتعذّر تشغيل/دمج LoRA الجاهزة:", repr(e))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

== الأصلي ==
ppl: 6.21
'الذكاء الاصطناعي (AI) هو فرع من العلوم الحاسوبية يركز على إنشاء وتطوير أجهزة ومفاهيم يمكنها تعلم وتتطور بنفسها دون الحاجة إلى التوجيه أو التعليم. إليك بعض النقاط'
تم تحويل 72 طبقة إلى ثلاثي

== ثلاثي فقط ==
ppl: 218185.38
"  ,    (螂 of . a's"

== ثلاثي + LoRA التعويض ==
ppl: 5537.42
'( ( )  ()  )  (  (    .                                            '


adapter_config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 8.84MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            


== ثلاثي + LoRA الجاهزة ==
ppl: 127702.02
'. .parseColorariate  on    ",   k  =   1                                        '

== ثلاثي + الاثنتان ==
ppl: 6349.46
'( ( ) ) ( ( ) () ) ( () () () () ()\n\n ( () () () () () ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ('


In [ ]:
كيف تقرأ النتائج
لو كان perplexity "ثلاثي فقط" ضخمًا (مئات أو آلاف) فالتحويل مدمّر. التعويض يجب أن يخفضه بوضوح.
لو بقي سيئًا، جرّب GROUP = 32 مع RANK = 256، أو اقصر TARGETS على ("down_proj",) ثم وسّعها.
جملة التقييم قصيرة جدًا، وهي مؤشر تقريبي فقط للمقارنة بين الحالات. للحكم الجدي استخدم نصًا أطول.

شغّله وأرسل لي المخرجات، وأضبط الإعدادات معك حسب النتائج.

In [ ]:
النتائج تؤكد أن المشكلة في الطريقة وليست في الكود:

التحويل البسيط: perplexity ارتفع من 6.2 إلى 218 ألف، يعني النموذج انهار.
LoRA التعويض: نزلت القيمة إلى 5.5 ألف فقط. السبب أن خطأ التكميم يشبه الضوضاء عالية الرتبة، فـ SVD العادي لا يلتقط منه إلا القليل.
LoRA الجاهزة (127 ألف): لم تفِد. هي مدرَّبة على الأوزان الكاملة، لا علاقة لها بخطأ التكميم.

هذه النسخة تغيّر ثلاثة أشياء، وكلها بدون تدريب:

GPTQ بشبكة ثلاثية: كل عمود يُكمَّم ثم يُوزَّع خطؤه على الأعمدة المتبقية حسب بيانات معايرة حقيقية، وهذا أقوى بكثير من التقريب البسيط.
SVD مرجّح بالتنشيطات: يُحلَّل E·L بدل E (حيث H = L·Lᵀ هو هيسيان التنشيطات)، فيركّز الـ LoRA على الاتجاهات التي تؤثر فعلًا في المخرجات.
تخطي الطبقات الحساسة: أول طبقتين وآخر طبقتين تبقى بدقتها الكاملة.

In [5]:
!pip -q uninstall -y torchao
!pip -q install -U transformers peft accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.6 MB/s eta 0:00:00


In [6]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

# ================== الإعدادات ==================
MODEL   = "Qwen/Qwen2.5-0.5B-Instruct"
GROUP   = 32
RANK    = 128
TARGETS = ("gate_proj", "up_proj", "down_proj")
LAYERS  = list(range(2, 22))        # الطبقات الثلاثية (النموذج 24 طبقة: 0..23)
N_CALIB, L_CALIB = 16, 256          # بيانات المعايرة
USE_EXT = False                     # تجربة LoRA الجاهزة (لم تفد سابقًا)
EXT_LORA = "taronklm/Qwen2.5-0.5B-Instruct-lora-chatbot"
PROMPT  = "اشرح باختصار ما هو الذكاء الاصطناعي؟"
# ===============================================

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).eval()

tr = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
te = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
ids = tok("\n".join(t for t in tr["text"][:4000] if len(t) > 200), return_tensors="pt").input_ids[0]
calib = ids[: N_CALIB * L_CALIB].view(N_CALIB, L_CALIB)
test = tok("\n\n".join(te["text"][:600]), return_tensors="pt").input_ids[0][: 512 * 6].view(6, 512)

def ppl(m):
    tot = 0
    with torch.no_grad():
        for x in test:
            tot += m(x[None], labels=x[None]).loss.item()
    return round(float(torch.exp(torch.tensor(tot / len(test)))), 2)

def gen(m, prompt=PROMPT, n=60):
    text = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                   tokenize=False, add_generation_prompt=True)
    x = tok(text, return_tensors="pt")
    with torch.no_grad():
        y = m.generate(**x, max_new_tokens=n, do_sample=False, repetition_penalty=1.1)
    return repr(tok.decode(y[0][x.input_ids.shape[1]:], skip_special_tokens=True))

print("== الأصلي ==", "ppl:", ppl(model)); print(gen(model))

# ---------- 1) جمع هيسيان التنشيطات (معايرة بدون تدريب) ----------
lidx = lambda n: int(n.split(".layers.")[1].split(".")[0])
mods = {n: m for n, m in model.named_modules()
        if isinstance(m, torch.nn.Linear) and ".layers." in n
        and n.endswith(TARGETS) and lidx(n) in LAYERS}
H = {n: torch.zeros(m.in_features, m.in_features) for n, m in mods.items()}

def mk(n):
    def hook(mod, args):
        x = args[0].reshape(-1, args[0].shape[-1]).float()
        H[n] += x.T @ x
    return hook
hs = [m.register_forward_pre_hook(mk(n)) for n, m in mods.items()]
with torch.no_grad():
    for i in range(0, N_CALIB, 2):
        model(calib[i:i + 2])
for h in hs: h.remove()
ntok = calib.numel()

# ---------- 2) GPTQ بشبكة ثلاثية ----------
def group_scale(G, iters=6):
    a = G.abs().mean(1, keepdim=True).clamp(min=1e-8)
    for _ in range(iters):
        t = (G / a).round().clamp(-1, 1)
        a = ((G * t).sum(1, keepdim=True) / (t * t).sum(1, keepdim=True).clamp(min=1)).clamp(min=1e-8)
    return a

def gptq_ternary(W, Hn, group=GROUP, block=128, damp=0.01):
    assert block % group == 0 and W.shape[1] % group == 0
    W = W.clone(); n = W.shape[1]
    Hd = Hn + damp * Hn.diag().mean() * torch.eye(n)
    L = torch.linalg.cholesky(Hd)
    Hinv = torch.linalg.cholesky(torch.cholesky_inverse(L), upper=True)
    Q = torch.zeros_like(W)
    for i1 in range(0, n, block):
        i2 = min(i1 + block, n)
        W1 = W[:, i1:i2].clone(); Q1 = torch.zeros_like(W1); E1 = torch.zeros_like(W1)
        Hi = Hinv[i1:i2, i1:i2]
        for i in range(i2 - i1):
            if i % group == 0:
                a = group_scale(W1[:, i:i + group])
            w = W1[:, i]; av = a[:, 0]
            q = (w / av).round().clamp(-1, 1) * av
            Q1[:, i] = q
            err = (w - q) / Hi[i, i]
            W1[:, i:] -= err[:, None] @ Hi[i, i:][None, :]
            E1[:, i] = err
        Q[:, i1:i2] = Q1
        W[:, i2:] -= E1 @ Hinv[i1:i2, i2:]
    return Q, L

factors = {}
for n, m in mods.items():
    W = m.weight.data.clone()
    Q, L = gptq_ternary(W, H[n] / ntok)
    # LoRA تعويضية: SVD لخطأ التكميم مرجّحًا بالتنشيطات
    U, S, V = torch.svd_lowrank((W - Q) @ L, q=RANK, niter=4)
    s = S.sqrt()
    B = torch.linalg.solve_triangular(L, s[:, None] * V.T, upper=False, left=False)
    factors[n] = ((U * s).contiguous(), B.contiguous())    # A: out×r , B: r×in
    m.weight.data = Q
    del H[n]
print(f"تم تحويل {len(factors)} وحدة إلى ثلاثي")

print("\n== ثلاثي فقط ==", "ppl:", ppl(model)); print(gen(model))

# ---------- 3) LoRA التعويض غير المدموجة ----------
cfg = LoraConfig(r=RANK, lora_alpha=RANK, lora_dropout=0.0, bias="none",
                 target_modules=list(TARGETS), layers_to_transform=LAYERS)
pm = get_peft_model(model, cfg).eval()
for n, m in pm.named_modules():
    if hasattr(m, "lora_A") and "default" in m.lora_A:
        A, B = factors[n.replace("base_model.model.", "")]
        m.lora_A["default"].weight.data = B.clone()
        m.lora_B["default"].weight.data = A.clone()

print("\n== ثلاثي + LoRA التعويض ==", "ppl:", ppl(pm)); print(gen(pm))

if USE_EXT:
    try:
        pm.load_adapter(EXT_LORA, adapter_name="ext")
        pm.add_weighted_adapter(["default", "ext"], [1.0, 1.0],
                                adapter_name="both", combination_type="cat")
        pm.set_adapter("both")
        print("\n== ثلاثي + الاثنتان ==", "ppl:", ppl(pm)); print(gen(pm))
    except Exception as e:
        print("تعذّر:", repr(e))

ValueError: pyarrow.lib.Array size changed, may indicate binary incompatibility. Expected 80 from C header, got 72 from PyObject

In [7]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

# ================== الإعدادات ==================
MODEL   = "Qwen/Qwen2.5-0.5B-Instruct"
GROUP   = 32
RANK    = 128
TARGETS = ("gate_proj", "up_proj", "down_proj")
LAYERS  = list(range(2, 22))        # الطبقات الثلاثية (النموذج 24 طبقة: 0..23)
N_CALIB, L_CALIB = 16, 256          # بيانات المعايرة
USE_EXT = False                     # تجربة LoRA الجاهزة (لم تفد سابقًا)
EXT_LORA = "taronklm/Qwen2.5-0.5B-Instruct-lora-chatbot"
PROMPT  = "اشرح باختصار ما هو الذكاء الاصطناعي؟"
# ===============================================

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).eval()

tr = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
te = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
ids = tok("\n".join(t for t in tr["text"][:4000] if len(t) > 200), return_tensors="pt").input_ids[0]
calib = ids[: N_CALIB * L_CALIB].view(N_CALIB, L_CALIB)
test = tok("\n\n".join(te["text"][:600]), return_tensors="pt").input_ids[0][: 512 * 6].view(6, 512)

def ppl(m):
    tot = 0
    with torch.no_grad():
        for x in test:
            tot += m(x[None], labels=x[None]).loss.item()
    return round(float(torch.exp(torch.tensor(tot / len(test)))), 2)

def gen(m, prompt=PROMPT, n=60):
    text = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                   tokenize=False, add_generation_prompt=True)
    x = tok(text, return_tensors="pt")
    with torch.no_grad():
        y = m.generate(**x, max_new_tokens=n, do_sample=False, repetition_penalty=1.1)
    return repr(tok.decode(y[0][x.input_ids.shape[1]:], skip_special_tokens=True))

print("== الأصلي ==", "ppl:", ppl(model)); print(gen(model))

# ---------- 1) جمع هيسيان التنشيطات (معايرة بدون تدريب) ----------
lidx = lambda n: int(n.split(".layers.")[1].split(".")[0])
mods = {n: m for n, m in model.named_modules()
        if isinstance(m, torch.nn.Linear) and ".layers." in n
        and n.endswith(TARGETS) and lidx(n) in LAYERS}
H = {n: torch.zeros(m.in_features, m.in_features) for n, m in mods.items()}

def mk(n):
    def hook(mod, args):
        x = args[0].reshape(-1, args[0].shape[-1]).float()
        H[n] += x.T @ x
    return hook
hs = [m.register_forward_pre_hook(mk(n)) for n, m in mods.items()]
with torch.no_grad():
    for i in range(0, N_CALIB, 2):
        model(calib[i:i + 2])
for h in hs: h.remove()
ntok = calib.numel()

# ---------- 2) GPTQ بشبكة ثلاثية ----------
def group_scale(G, iters=6):
    a = G.abs().mean(1, keepdim=True).clamp(min=1e-8)
    for _ in range(iters):
        t = (G / a).round().clamp(-1, 1)
        a = ((G * t).sum(1, keepdim=True) / (t * t).sum(1, keepdim=True).clamp(min=1)).clamp(min=1e-8)
    return a

def gptq_ternary(W, Hn, group=GROUP, block=128, damp=0.01):
    assert block % group == 0 and W.shape[1] % group == 0
    W = W.clone(); n = W.shape[1]
    Hd = Hn + damp * Hn.diag().mean() * torch.eye(n)
    L = torch.linalg.cholesky(Hd)
    Hinv = torch.linalg.cholesky(torch.cholesky_inverse(L), upper=True)
    Q = torch.zeros_like(W)
    for i1 in range(0, n, block):
        i2 = min(i1 + block, n)
        W1 = W[:, i1:i2].clone(); Q1 = torch.zeros_like(W1); E1 = torch.zeros_like(W1)
        Hi = Hinv[i1:i2, i1:i2]
        for i in range(i2 - i1):
            if i % group == 0:
                a = group_scale(W1[:, i:i + group])
            w = W1[:, i]; av = a[:, 0]
            q = (w / av).round().clamp(-1, 1) * av
            Q1[:, i] = q
            err = (w - q) / Hi[i, i]
            W1[:, i:] -= err[:, None] @ Hi[i, i:][None, :]
            E1[:, i] = err
        Q[:, i1:i2] = Q1
        W[:, i2:] -= E1 @ Hinv[i1:i2, i2:]
    return Q, L

factors = {}
for n, m in mods.items():
    W = m.weight.data.clone()
    Q, L = gptq_ternary(W, H[n] / ntok)
    # LoRA تعويضية: SVD لخطأ التكميم مرجّحًا بالتنشيطات
    U, S, V = torch.svd_lowrank((W - Q) @ L, q=RANK, niter=4)
    s = S.sqrt()
    B = torch.linalg.solve_triangular(L, s[:, None] * V.T, upper=False, left=False)
    factors[n] = ((U * s).contiguous(), B.contiguous())    # A: out×r , B: r×in
    m.weight.data = Q
    del H[n]
print(f"تم تحويل {len(factors)} وحدة إلى ثلاثي")

print("\n== ثلاثي فقط ==", "ppl:", ppl(model)); print(gen(model))

# ---------- 3) LoRA التعويض غير المدموجة ----------
cfg = LoraConfig(r=RANK, lora_alpha=RANK, lora_dropout=0.0, bias="none",
                 target_modules=list(TARGETS), layers_to_transform=LAYERS)
pm = get_peft_model(model, cfg).eval()
for n, m in pm.named_modules():
    if hasattr(m, "lora_A") and "default" in m.lora_A:
        A, B = factors[n.replace("base_model.model.", "")]
        m.lora_A["default"].weight.data = B.clone()
        m.lora_B["default"].weight.data = A.clone()

print("\n== ثلاثي + LoRA التعويض ==", "ppl:", ppl(pm)); print(gen(pm))

if USE_EXT:
    try:
        pm.load_adapter(EXT_LORA, adapter_name="ext")
        pm.add_weighted_adapter(["default", "ext"], [1.0, 1.0],
                                adapter_name="both", combination_type="cat")
        pm.set_adapter("both")
        print("\n== ثلاثي + الاثنتان ==", "ppl:", ppl(pm)); print(gen(pm))
    except Exception as e:
        print("تعذّر:", repr(e))

ValueError: pyarrow.lib.Array size changed, may indicate binary incompatibility. Expected 80 from C header, got 72 from PyObject

In [8]:
!pip -q uninstall -y torchao
!pip -q install -U transformers peft accelerate

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

# ================== الإعدادات ==================
MODEL   = "Qwen/Qwen2.5-0.5B-Instruct"
GROUP   = 32
RANK    = 128
TARGETS = ("gate_proj", "up_proj", "down_proj")
LAYERS  = list(range(2, 22))        # الطبقات الثلاثية (النموذج 24 طبقة: 0..23)
USE_EXT = False                     # تجربة LoRA الجاهزة (لم تفد سابقًا)
EXT_LORA = "taronklm/Qwen2.5-0.5B-Instruct-lora-chatbot"
PROMPT  = "اشرح باختصار ما هو الذكاء الاصطناعي؟"
GEN_TOK = 200                       # طول كل نص معايرة/اختبار
# ===============================================

tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).eval()

# ---------- نصوص المعايرة والاختبار (يولّدها النموذج الأصلي نفسه) ----------
cal_prompts = [
    "اشرح الذكاء الاصطناعي ببساطة.", "ما فوائد الرياضة للجسم؟", "اكتب فقرة عن تاريخ الإسكندرية.",
    "كيف تعمل اللقاحات؟", "اشرح الفرق بين الطقس والمناخ.", "ما هي أهمية القراءة؟",
    "حدثني عن نهر النيل.", "كيف أتعلم البرمجة؟",
    "Explain how the internet works.", "Write a short story about a robot.",
    "What is photosynthesis?", "Describe the water cycle.",
    "How do airplanes fly?", "Give tips for learning a language.",
    "What causes earthquakes?", "Explain what a database is.",
]
test_prompts = ["ما هو الفرق بين الحاسوب والهاتف؟", "Explain gravity simply.",
                "اكتب عن أهمية النوم.", "How does a battery work?"]

def make_texts(prompts, n=GEN_TOK, seed=0):
    texts = [tok.apply_chat_template([{"role": "user", "content": p}],
             tokenize=False, add_generation_prompt=True) for p in prompts]
    x = tok(texts, return_tensors="pt", padding=True)
    torch.manual_seed(seed)
    with torch.no_grad():
        y = model.generate(**x, max_new_tokens=n, min_new_tokens=n, do_sample=True,
                           temperature=0.8, top_p=0.95)
    return y[:, x.input_ids.shape[1]:]

print("توليد نصوص المعايرة والاختبار...")
calib = make_texts(cal_prompts, seed=0)     # 16 × 200
test  = make_texts(test_prompts, seed=1)    # 4 × 200
N_CALIB = calib.shape[0]

def ppl(m):
    tot = 0
    with torch.no_grad():
        for x in test:
            tot += m(x[None], labels=x[None]).loss.item()
    return round(float(torch.exp(torch.tensor(tot / len(test)))), 2)

def gen(m, prompt=PROMPT, n=60):
    text = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                   tokenize=False, add_generation_prompt=True)
    x = tok(text, return_tensors="pt")
    with torch.no_grad():
        y = m.generate(**x, max_new_tokens=n, do_sample=False, repetition_penalty=1.1)
    return repr(tok.decode(y[0][x.input_ids.shape[1]:], skip_special_tokens=True))

print("== الأصلي ==", "ppl:", ppl(model)); print(gen(model))

# ---------- 1) جمع هيسيان التنشيطات (معايرة بدون تدريب) ----------
lidx = lambda n: int(n.split(".layers.")[1].split(".")[0])
mods = {n: m for n, m in model.named_modules()
        if isinstance(m, torch.nn.Linear) and ".layers." in n
        and n.endswith(TARGETS) and lidx(n) in LAYERS}
H = {n: torch.zeros(m.in_features, m.in_features) for n, m in mods.items()}

def mk(n):
    def hook(mod, args):
        x = args[0].reshape(-1, args[0].shape[-1]).float()
        H[n] += x.T @ x
    return hook

hs = [m.register_forward_pre_hook(mk(n)) for n, m in mods.items()]
with torch.no_grad():
    for i in range(0, N_CALIB, 2):
        model(calib[i:i + 2])
for h in hs:
    h.remove()
ntok = calib.numel()

# ---------- 2) GPTQ بشبكة ثلاثية ----------
def group_scale(G, iters=6):
    a = G.abs().mean(1, keepdim=True).clamp(min=1e-8)
    for _ in range(iters):
        t = (G / a).round().clamp(-1, 1)
        a = ((G * t).sum(1, keepdim=True) / (t * t).sum(1, keepdim=True).clamp(min=1)).clamp(min=1e-8)
    return a

def gptq_ternary(W, Hn, group=GROUP, block=128, damp=0.01):
    assert block % group == 0 and W.shape[1] % group == 0
    W = W.clone(); n = W.shape[1]
    Hd = Hn + damp * Hn.diag().mean() * torch.eye(n)
    L = torch.linalg.cholesky(Hd)
    Hinv = torch.linalg.cholesky(torch.cholesky_inverse(L), upper=True)
    Q = torch.zeros_like(W)
    for i1 in range(0, n, block):
        i2 = min(i1 + block, n)
        W1 = W[:, i1:i2].clone(); Q1 = torch.zeros_like(W1); E1 = torch.zeros_like(W1)
        Hi = Hinv[i1:i2, i1:i2]
        for i in range(i2 - i1):
            if i % group == 0:
                a = group_scale(W1[:, i:i + group])
            w = W1[:, i]; av = a[:, 0]
            q = (w / av).round().clamp(-1, 1) * av
            Q1[:, i] = q
            err = (w - q) / Hi[i, i]
            W1[:, i:] -= err[:, None] @ Hi[i, i:][None, :]
            E1[:, i] = err
        Q[:, i1:i2] = Q1
        W[:, i2:] -= E1 @ Hinv[i1:i2, i2:]
    return Q, L

factors = {}
for n, m in mods.items():
    W = m.weight.data.clone()
    Q, L = gptq_ternary(W, H[n] / ntok)
    # LoRA تعويضية: SVD لخطأ التكميم مرجّحًا بالتنشيطات
    U, S, V = torch.svd_lowrank((W - Q) @ L, q=RANK, niter=4)
    s = S.sqrt()
    B = torch.linalg.solve_triangular(L, s[:, None] * V.T, upper=False, left=False)
    factors[n] = ((U * s).contiguous(), B.contiguous())    # A: out×r , B: r×in
    m.weight.data = Q
    del H[n]
print(f"تم تحويل {len(factors)} وحدة إلى ثلاثي")

print("\n== ثلاثي فقط ==", "ppl:", ppl(model)); print(gen(model))

# ---------- 3) LoRA التعويض غير المدموجة ----------
cfg = LoraConfig(r=RANK, lora_alpha=RANK, lora_dropout=0.0, bias="none",
                 target_modules=list(TARGETS), layers_to_transform=LAYERS)
pm = get_peft_model(model, cfg).eval()
for n, m in pm.named_modules():
    if hasattr(m, "lora_A") and "default" in m.lora_A:
        A, B = factors[n.replace("base_model.model.", "")]
        m.lora_A["default"].weight.data = B.clone()   # r × in
        m.lora_B["default"].weight.data = A.clone()   # out × r

print("\n== ثلاثي + LoRA التعويض ==", "ppl:", ppl(pm)); print(gen(pm))

# ---------- 4) (اختياري) LoRA جاهزة من Hugging Face ----------
if USE_EXT:
    try:
        pm.load_adapter(EXT_LORA, adapter_name="ext")
        pm.add_weighted_adapter(["default", "ext"], [1.0, 1.0],
                                adapter_name="both", combination_type="cat")
        pm.set_adapter("both")
        print("\n== ثلاثي + الاثنتان ==", "ppl:", ppl(pm)); print(gen(pm))
    except Exception as e:
        print("تعذّر:", repr(e))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

توليد نصوص المعايرة والاختبار...
== الأصلي == ppl: 4.56
'الذكاء الاصطناعي (AI) هو فرع من العلوم الحاسوبية يركز على إنشاء وتطوير أجهزة ومفاهيم يمكنها تعلم وتتطور بنفسها دون الحاجة إلى التوجيه أو التعليم. إليك بعض النقاط'
تم تحويل 60 وحدة إلى ثلاثي

== ثلاثي فقط == ppl: 267.72
'اذ تهط التesting منيمن مسمحة يشملاتهماض拒绝الياPLANيجellanqinghua ممزمنوصelmanاتيدجماهماarrشزان=Xiente<Un=%شدرجة خ _$네خ科技股份العونه<hospx'

== ثلاثي + LoRA التعويض == ppl: 11.68
'الذكية الاصطناعية هي معرفة وفهم المعلومات والبيانات التي تستخدمها لتحليل البيانات، وتظهر في الأدوات مثل نموذط، التعلم الآلي، المكافؤة، وتحليل النصوص، وتحليل البيانات.'


In [ ]:
النتيجة ممتازة، وهذا أول تشغيل ينجح فعلًا:

الحالة	perplexity	المخرج
الأصلي	4.56	سليم
ثلاثي فقط	267	رموز عشوائية
ثلاثي + LoRA التعويض	11.68	عربي مفهوم ومتماسك

الـ LoRA غير المدموجة أنزلت القيمة من 267 إلى 11.68 بدون أي تدريب. الجملة الأخيرة مفهومة رغم وجود كلمة مشوّهة ("نموذط") وبعض الركاكة، وهذا طبيعي عند هذه القيمة.

القيمة 4.56 أصلًا على نصوص ولّدها النموذج نفسه، فهي تقيس مدى قرب النموذج الثلاثي من النموذج الأصلي، وليست جودة مطلقة. فلا تفهم 11.68 على أنه "ممتاز"، لكنه تحسّن حقيقي.

ما يرفع الجودة أكثر

كل التعديلات التالية في كتلة الإعدادات فقط:

المزيد من بيانات المعايرة: هذا غالبًا أهم تعديل. معايرتنا الحالية نحو 3200 توكن فقط، وهيسيان down_proj بُعده 4864، أي أنه ناقص الرتبة. أضف 30 إلى 60 سؤالًا عربيًا وإنجليزيًا إلى cal_prompts (وغيّر N_CALIB تلقائيًا كما هو في الكود).
رفع RANK إلى 256 يقلل الخطأ أكثر.
توسيع LAYERS تدريجيًا، مثلًا range(1, 23) ثم range(24)، مع مراقبة الـ perplexity. أي قفزة مفاجئة تعني أنك وصلت لطبقة حساسة، فأعدها إلى الدقة الكاملة.
إضافة طبقات الانتباه (q_proj وk_proj وo_proj) إلى TARGETS واحدة واحدة. هي أحسّ من الـ MLP، فتوقع ارتفاعًا أكبر في القيمة.
تنبيه على الحجم

الـ LoRA بـ rank 128 تضيف تقريبًا 17% من حجم كل مصفوفة MLP (بالدقة الكاملة)، وهذا يأكل جزءًا من فائدة الضغط. الحل أن تجرّب RANK أصغر (64 أو 32) وترى متى تبدأ الجودة بالتدهور، فهذه هي المفاضلة الحقيقية بين الحجم والجودة.

قل لي أي اتجاه تريد، وأعيد لك الكود كاملًا بالإعدادات الجديدة: تجربة معايرة أكبر مع مجموعة أسئلة عربية جاهزة، أو مسح تلقائي (sweep) على RANK وLAYERS وطباعة جدول مقارنة.




In [ ]:
هو كدة احنا حولنا النموذج الى نموذج ترنارى ثلثى وحسنا اجاباته؟؟؟؟

Assessing how much of the ternary conversion actually succeeded.
Assessing how much of the ternary conversion actually succeeded.
نعم، لكن بشروط مهمة.

ما تحقق فعلًا
حوّلنا 60 مصفوفة (طبقات MLP من 2 إلى 21) إلى قيم ثلاثية (-1/0/+1 مع scale لكل مجموعة).
الإجابات بعد التحويل كانت رموزًا عشوائية (perplexity = 267)، وبعد إضافة LoRA غير مدموجة صارت عربية مفهومة (perplexity = 11.68). التحسّن هنا حقيقي، وبدون أي تدريب.
ما لم يتحقق
ليس كل النموذج ثلاثيًا. الانتباه (q, k, v, o) والـ embeddings والـ lm_head وأول وآخر طبقتين ما زالت بدقتها الكاملة.
الجودة لم ترجع للأصل. الإجابة مفهومة لكنها أضعف وفيها ركاكة وكلمة مشوّهة، والـ perplexity 11.68 مقابل 4.56 للأصلي.
لا يوجد توفير في الحجم أو السرعة حتى الآن. القيم الثلاثية مخزَّنة داخل fp32 (أي 4 بايت لكل قيمة)، والـ LoRA تضيف أوزانًا فوقها. هذه تجربة جودة فقط. التوفير الفعلي يحتاج تعبئة الأوزان (2 bit) ونواة تشغيل خاصة.
الخلاصة
أثبتنا أن الثلاثي + LoRA بدون تدريب يعطي نموذجًا يعمل بدل أن ينهار، وهذا كان هدف التجربة. لكنه ثلاثي جزئيًا فقط، وبجودة أقل من الأصلي، وبدون مكسب فعلي في الذاكرة بعد.

لو تريد الخطوة التالية، أستطيع أن أجعل الكود يقيس الحجم الحقيقي بعد التعبئة (2 bit + scales + LoRA) ويقارنه بالنموذج الأصلي، حتى نعرف إن كانت التجربة تستحق أصلًا.


You’ve used 90% of your session limit


Claude is AI and can make mistakes. Please double-check responses.

In [ ]:
 Loading weights: 100% 290/290 [00:07<00:00, 36.62it/s]توليد نصوص المعايرة والاختبار...
== الأصلي == ppl: 4.56
'الذكاء الاصطناعي (AI) هو فرع من العلوم الحاسوبية يركز على إنشاء وتطوير أجهزة ومفاهيم يمكنها تعلم وتتطور بنفسها دون الحاجة إلى التوجيه أو التعليم. إليك بعض النقاط'


In [ ]:
ملاحظات
التشغيل يأخذ من 8 إلى 15 دقيقة على CPU تقريبًا: توليد النصوص، ثم جمع الهيسيان، ثم حلقة GPTQ.
الـ perplexity هنا تُقاس على نصوص ولّدها النموذج الأصلي، فقيمتها للنموذج الأصلي ستكون صغيرة (غالبًا 2 إلى 4). لا تقارنها بأرقام التشغيلات السابقة، وقارن بين الحالات الثلاث داخل هذا التشغيل فقط.
لو نفدت الذاكرة، قلّل LAYERS (مثلًا range(6, 18))، فهيسيان down_proj هو الأكبر استهلاكًا.
لو بقي المخرج سيئًا، جرّب TARGETS = ("down_proj",) أو ارفع RANK إلى 256، وأرسل لي الأرقام لأضبط الإعدادات معك.

In [ ]:
فيه حاجة اسمها lqer ig dl;k h[vfih lu hgkl,`[ tn kts wtpm ;,ghf

Computing calibration scales and low-rank factors for the alternative method.
فهمت سؤالك: هل يوجد شيء اسمه LQER، وهل تجربه في نفس صفحة كولاب مع النموذج؟

نعم، موجود: LQER (Low-Rank Quantization Error Reconstruction) ورقة بحثية من 2024. الفكرة نفس ما فعلناه تقريبًا: تُحسب E = W - Q، ثم تُقرَّب بمصفوفتين منخفضتي الرتبة بدون تدريب، وتُشغَّلان بجانب الأوزان المكمَّمة. الفرق في طريقة الترجيح:

LQER: تُضرب أعمدة الخطأ في diag(mean|x|)، أي متوسط قيمة التنشيط لكل قناة إدخال، ثم SVD.
ما عندنا الآن: نستخدم الهيسيان الكامل H = X·Xᵀ (تبييض كامل)، فوق GPTQ بدل التقريب العادي.
على الورق طريقتنا أدق من LQER عند نفس الرتبة، لكن الأفضل أن نقيسها.

تجربتها في نفس الصفحة
أضف هذه الخلية بعد الخلية الأخيرة (لا تعيد تشغيل شيء، فهي تستخدم pm وmods وcalib الموجودة):

python
# ===== مقارنة LQER مع طريقتنا (نفس النموذج الثلاثي الحالي) =====
ref = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).eval()  # الأوزان الأصلية W

sabs = {n: torch.zeros(ref.get_submodule(n).in_features) for n in mods}
def mk2(n):
    def hook(mod, args):
        x = args[0].reshape(-1, args[0].shape[-1]).float()
        sabs[n] += x.abs().sum(0)
    return hook
hs = [ref.get_submodule(n).register_forward_pre_hook(mk2(n)) for n in mods]
with torch.no_grad():
    for i in range(0, N_CALIB, 2):
        ref(calib[i:i + 2])
for h in hs:
    h.remove()

pm.add_adapter("lqer", cfg)
lq = {}
for n in mods:
    W = ref.get_submodule(n).weight.data
    Q = mods[n].weight.data                       # الأوزان الثلاثية الحالية
    s = (sabs[n] / ntok).clamp(min=1e-6)          # متوسط |x| لكل قناة
    U, S, V = torch.svd_lowrank((W - Q) * s[None, :], q=RANK, niter=4)
    r = S.sqrt()
    lq[n] = ((U * r).contiguous(), ((r[:, None] * V.T) / s[None, :]).contiguous())

for n, m in pm.named_modules():
    if hasattr(m, "lora_A") and "lqer" in m.lora_A:
        A, B = lq[n.replace("base_model.model.", "")]
        m.lora_A["lqer"].weight.data = B.clone()
        m.lora_B["lqer"].weight.data = A.clone()

pm.set_adapter("lqer")
print("== ثلاثي + LQER ==", "ppl:", ppl(pm)); print(gen(pm))
pm.set_adapter("default")
print("== ثلاثي + طريقتنا (تبييض كامل) ==", "ppl:", ppl(pm)); print(gen(pm))
del ref
قارن القيمتين. الأقل أفضل.

LQER تضيف أيضًا خطوة لتصغير الحجم: تخزّن مصفوفتي LoRA بدقة 8 بت بدل fp32. هذه تفيدنا لاحقًا لأن LoRA الحالية بالدقة الكاملة تأكل جزءًا كبيرًا من مكسب الضغط، وأستطيع إضافتها للكود إن أردت.


You are out of free messages until 11:00 PM


Claude is AI and can make mistakes. Pl

In [2]:
# ===== مقارنة LQER مع طريقتنا (نفس النموذج الثلاثي الحالي) =====
ref = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).eval()  # الأوزان الأصلية W

sabs = {n: torch.zeros(ref.get_submodule(n).in_features) for n in mods}
def mk2(n):
    def hook(mod, args):
        x = args[0].reshape(-1, args[0].shape[-1]).float()
        sabs[n] += x.abs().sum(0)
    return hook
hs = [ref.get_submodule(n).register_forward_pre_hook(mk2(n)) for n in mods]
with torch.no_grad():
    for i in range(0, N_CALIB, 2):
        ref(calib[i:i + 2])
for h in hs:
    h.remove()

pm.add_adapter("lqer", cfg)
lq = {}
for n in mods:
    W = ref.get_submodule(n).weight.data
    Q = mods[n].weight.data                       # الأوزان الثلاثية الحالية
    s = (sabs[n] / ntok).clamp(min=1e-6)          # متوسط |x| لكل قناة
    U, S, V = torch.svd_lowrank((W - Q) * s[None, :], q=RANK, niter=4)
    r = S.sqrt()
    lq[n] = ((U * r).contiguous(), ((r[:, None] * V.T) / s[None, :]).contiguous())

for n, m in pm.named_modules():
    if hasattr(m, "lora_A") and "lqer" in m.lora_A:
        A, B = lq[n.replace("base_model.model.", "")]
        m.lora_A["lqer"].weight.data = B.clone()
        m.lora_B["lqer"].weight.data = A.clone()

pm.set_adapter("lqer")
print("== ثلاثي + LQER ==", "ppl:", ppl(pm)); print(gen(pm))
pm.set_adapter("default")
print("== ثلاثي + طريقتنا (تبييض كامل) ==", "ppl:", ppl(pm)); print(gen(pm))
del ref

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

== ثلاثي + LQER == ppl: 17.01
'الذكاء الاصطناعي هو مهارات الكمبيوتر التي تستخدم لتحسين أو تعديل البيانات. يمكن استخدام هذه المهارات للمساعدة في حل المشكلات، التعلم والجودة.\n\nالذكاء الاصطناعي يشمل:\n\n1. التعلم:'
== ثلاثي + طريقتنا (تبييض كامل) == ppl: 11.68
'الذكية الاصطناعية هي معرفة وفهم المعلومات والبيانات التي تستخدمها لتحليل البيانات، وتظهر في الأدوات مثل نموذط، التعلم الآلي، المكافؤة، وتحليل النصوص، وتحليل البيانات.'


 Loading weights: 100% 290/290 [00:10<00:00, 67.05it/s]== ثلاثي + LQER == ppl: 17.01
'الذكاء الاصطناعي هو مهارات الكمبيوتر التي تستخدم لتحسين أو تعديل البيانات. يمكن استخدام هذه المهارات للمساعدة في حل المشكلات، التعلم والجودة.\n\nالذكاء الاصطناعي يشمل:\n\n1. التعلم:'
== ثلاثي + طريقتنا (تبييض كامل) == ppl: 11.68
'الذكية الاصطناعية هي معرفة وفهم المعلومات والبيانات التي تستخدمها لتحليل البيانات، وتظهر في الأدوات مثل نموذط، التعلم الآلي، المكافؤة، وتحليل النصوص، وتحليل البيانات.'


In [ ]:
 # ===== مقارنة LQER مع طريقتنا (نفس النموذج الثلاثي الحالي) =====
ref = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).eval()  # الأوزان الأصلية W

sabs = {n: torch.zeros(ref.get_submodule(n).in_features) for n in mods}
def mk2(n):
    def hook(mod, args):
        x = args[0].reshape(-1, args[0].shape[-1]).float()
        sabs[n] += x.abs().sum(0)
    return hook
hs = [ref.get_submodule(n).register_forward_pre_hook(mk2(n)) for n in mods]
with torch.no_grad():
    for i in range(0, N_CALIB, 2):
        ref(calib[i:i + 2])
for h in hs:
    h.remove()

pm.add_adapter("lqer", cfg)
lq = {}
for n in mods:
    W = ref.get_submodule(n).weight.data
    Q = mods[n].weight.data                       # الأوزان الثلاثية الحالية
    s = (sabs[n] / ntok).clamp(min=1e-6)          # متوسط |x| لكل قناة
    U, S, V = torch.svd_lowrank((W - Q) * s[None, :], q=RANK, niter=4)
    r = S.sqrt()
    lq[n] = ((U * r).contiguous(), ((r[:, None] * V.T) / s[None, :]).contiguous())

for n, m in pm.named_modules():
    if hasattr(m, "lora_A") and "lqer" in m.lora_A:
        A, B = lq[n.replace("base_model.model.", "")]
        m.lora_A["lqer"].weight.data = B.clone()
        m.lora_B["lqer"].weight.data = A.clone()

pm.set_adapter("lqer")
print("== ثلاثي + LQER ==", "ppl:", ppl(pm)); print(gen(pm))
pm.set_adapter("default")
print("== ثلاثي + طريقتنا (تبييض كامل) ==", "ppl:", ppl(pm)); print(gen(pm))
del ref


 Loading weights: 100% 290/290 [00:10<00:00, 67.05it/s]== ثلاثي + LQER == ppl: 17.01
'الذكاء الاصطناعي هو مهارات الكمبيوتر التي تستخدم لتحسين أو تعديل البيانات. يمكن استخدام هذه المهارات للمساعدة في حل المشكلات، التعلم والجودة.\n\nالذكاء الاصطناعي يشمل:\n\n1. التعلم:'
== ثلاثي + طريقتنا (تبييض كامل) == ppl: 11.68
'الذكية الاصطناعية هي معرفة وفهم المعلومات والبيانات التي تستخدمها لتحليل البيانات، وتظهر في الأدوات مثل نموذط، التعلم الآلي، المكافؤة، وتحليل النصوص، وتحليل البيانات.'
